# Celestack Workflow

This is a development workflow for the `celestack` project.

Before release, it will be fully substituted by a CLI, TUI, or GUI.

## Project-level Settings:

In [ ]:
from pathlib import Path

from celestack.stack import FrameStack

PROJECT = "test"

## Initialize the Project Stack

In [ ]:
stack = FrameStack(PROJECT)
stack

## Load the Frames

In [ ]:
stack = FrameStack.from_state(PROJECT)
lf_paths = list(Path("/home/martin/Desktop/tenerife/lf").glob("*.tif"))
df_paths = list(Path("/home/martin/Desktop/tenerife/df").glob("*.tif"))

stack.load_frames(
    lf_paths=lf_paths,
    df_paths=df_paths,
)

stack = FrameStack.from_state(PROJECT)
print(f"{len(stack.light_frames)=}, {len(stack.dark_frames)=}")

## Apply the Dark Frames Correction

In [ ]:
stack = FrameStack.from_state(PROJECT)

stack.apply_dark_frames_correction()

stack = FrameStack.from_state(PROJECT)
stack.master_dark

## Create the Average Light Frame

In [ ]:
stack = FrameStack.from_state(PROJECT)

stack.create_average_light_frame()

stack = FrameStack.from_state(PROJECT)
stack.avg_light

## Create and Apply the Foreground Mask

Start with clustering the pixels of the average light frame.

In [ ]:
stack = FrameStack.from_state(PROJECT)
if stack.avg_light is None:
    raise ValueError("No average light frame found.")

alf = stack.avg_light
alf.cluster_pixels(n_clusters=2)
fig = alf.plot_clusters()
fig.show()


Now, turn the clustered pixels into a `Mask` object and add it to the stack.

In [ ]:
alf.initialize_mask(foreground_cluster_labels=[0])

# Explicitly mask/unmask some areas of the image
alf.set_mask_in_box(False, y2=1760)  # top part of the image is sky
alf.set_mask_in_box(True, y1=2035)  # bottom part of the image is foreground

mask = alf.create_mask()

stack.add_mask(mask)

And finally, add the mask to the stack.

In [ ]:


# Show that the mask has been added to the stack
stack = FrameStack.from_state(PROJECT)
if stack.mask is None or stack.light_frames["P3290058"].mask is None:
    raise ValueError("No mask found in stack or light frame.")

stack.mask.plot().show()

## Sky Segmentation

In [ ]:
stack = FrameStack.from_state(PROJECT)

stack.segment_sky()

stack = FrameStack.from_state(PROJECT)
if not stack.segment_boxes:
    raise ValueError("No segment boxes found in stack.")

stack = FrameStack.from_state(PROJECT)
stack.plot().show()

## Detect Stars in Reference Frame

First, set the reference frame.

In [ ]:
stack = FrameStack.from_state(PROJECT)

stack.set_reference_frame("P3290100")

Now, detect stars in the reference frame.

In [ ]:
stack = FrameStack.from_state(PROJECT)
stack.detect_stars_in_ref_frame(n=1000)

stack.stars_table

Let's plot the stars.

In [ ]:
stack = FrameStack.from_state(PROJECT)

fig = stack.plot()
fig.show()

stack.stars_table

## Propagate the stars across the stack

Let's test the algo for propagating a single star across the stack.

In [ ]:
import pandas as pd

import plotly.express as px

assert stack.stars_table is not None
assert stack.ref_frame is not None

ref_name = stack.ref_frame.name
# star_id = 900
star_id = 75

star_coordinates = pd.DataFrame(index=list(stack.light_frames), columns=["x", "y"])
x0, y0 = stack.stars_table.at[star_id, "x"], stack.stars_table.at[star_id, "y"]
fwhm = stack.stars_table.at[star_id, "fwhm"]
threshold = stack.stars_table.at[star_id, "threshold"]

star_coordinates.loc[ref_name, ["x", "y"]] = [x0, y0]

# TODO: Implement a better prediction of the star position in the next frame
# TODO: Silence the warnings

# Go down the stack:
x, y = x0, y0
for frame_name in star_coordinates.loc[ref_name:, :].index:
    frame = stack.light_frames[frame_name]
    star = frame.find_star(x, y, fwhm, threshold - 0.2, 1.7, 1.5)
    if star is not None:
        star_coordinates.loc[frame_name, ["x", "y"]] = [star["x"], star["y"]]
        x, y = star["x"], star["y"]

# Go up the stack:
x, y = x0, y0
for frame_name in star_coordinates.loc[:ref_name, :].index[::-1]:
    frame = stack.light_frames[frame_name]
    star = frame.find_star(x, y, fwhm, threshold - 0.2, 1.7, 1.5)
    if star is not None:
        star_coordinates.loc[frame_name, ["x", "y"]] = [star["x"], star["y"]]
        x, y = star["x"], star["y"]

In [ ]:
# Let's plot the star coordinates across the stack:
fig = px.scatter(
    star_coordinates,
    x="x",
    y="y",
    color=star_coordinates.index,
    title=f"Star {star_id} coordinates across the stack",
)
fig.update_traces(marker=dict(size=10))
fig.update_layout(
    xaxis_title="X coordinate",
    yaxis_title="Y coordinate",
    legend_title="Frame",
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1,
    yaxis_autorange='reversed',  # Invert the Y axis
)
fig.show()

star_coordinates